# Multivariate Calculus for Machine Learning
Companion to the calculus notes. Every worked example, checked numerically.

## 1. A derivative is a limit: watch the secant slope approach $2x$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f = lambda x: x**2
x = 3.0
for h in [1, 0.1, 0.01, 0.001, 1e-6]:
    print(f"h={h:<8} slope={(f(x + h) - f(x)) / h:.6f}")
print("exact derivative 2x =", 2 * x)

## 2. Stationary points of $x^4 - 12x^3 + 36x^2$

In [ ]:
f = lambda x: x**4 - 12*x**3 + 36*x**2
d1 = lambda x: 4*x**3 - 36*x**2 + 72*x
d2 = lambda x: 12*x**2 - 72*x + 72
for x0 in [0, 3, 6]:
    print(f"x={x0}: f={f(x0)}, f'={d1(x0)}, f''={d2(x0)}")
xs = np.linspace(-1, 7, 300)
plt.plot(xs, f(xs)); plt.scatter([0, 3, 6], [f(0), f(3), f(6)], color="red")
plt.title("two minima (x=0, x=6) and a maximum (x=3)")

## 3. Gradient descent on one weight
Model $\hat y = w x$ on the points (1,2), (2,3), (3,5). Exact answer $w^* = 23/14$.

In [ ]:
x = np.array([1, 2, 3.]); y = np.array([2, 3, 5.])
C = lambda w: ((y - w * x) ** 2).sum()
dC = lambda w: -2 * (x * (y - w * x)).sum()

def descend(eta, steps=6, w=0.0):
    print(f"eta = {eta}")
    for t in range(steps):
        g = dC(w); w = w - eta * g
        print(f"  step {t+1}: grad={g:9.3f}  w={w:8.4f}  cost={C(w):9.4f}")
    return w

descend(0.02)
descend(0.08)          # too large: 0.08 > 2 / C'' = 2 / 28
print("exact w* =", 23 / 14)

## 4. Taylor approximation of $x^5 - x^4 + 2$ around 1

In [ ]:
f = lambda x: x**5 - x**4 + 2
P2 = lambda x: 2 + (x - 1) + 4 * (x - 1)**2
for x0 in [1.05, 1.2, 1.5]:
    print(f"x={x0}: exact={f(x0):.5f}  quadratic={P2(x0):.5f}  rel. error={abs(f(x0) - P2(x0)) / f(x0):.4%}")

## 5. Numerical gradient check

In [ ]:
def f(w):
    return 2*w[0]**2 + w[1]**2 + 2*w[0]*w[1] - 6*w[0] - 4*w[1]

def grad_f(w):
    return np.array([4*w[0] + 2*w[1] - 6, 2*w[0] + 2*w[1] - 4])

def numerical_grad(f, w, h=1e-5):
    g = np.zeros_like(w)
    for j in range(len(w)):
        e = np.zeros_like(w); e[j] = h
        g[j] = (f(w + e) - f(w - e)) / (2 * h)
    return g

w = np.array([0.3, -1.2])
print(grad_f(w), numerical_grad(f, w))

## 6. Newton vs. gradient descent on the 2-variable quadratic

In [ ]:
H = np.array([[4., 2.], [2., 2.]])
print("Hessian eigenvalues:", np.linalg.eigvalsh(H))

w = np.zeros(2)
w_newton = w - np.linalg.solve(H, grad_f(w))
print("Newton, one step:", w_newton, " f =", f(w_newton))

w = np.zeros(2); path = [w.copy()]
while np.linalg.norm(w - 1) > 1e-3:
    w = w - 0.1 * grad_f(w); path.append(w.copy())
print("gradient descent: steps to get within 0.001 of (1,1):", len(path) - 1)

path = np.array(path)
g1, g2 = np.meshgrid(np.linspace(-0.5, 2, 100), np.linspace(-0.5, 2, 100))
plt.contour(g1, g2, f(np.array([g1, g2])), levels=20)
plt.plot(path[:, 0], path[:, 1], "o-", ms=3, label="gradient descent")
plt.plot([0, 1], [0, 1], "r--", label="Newton (one step)")
plt.legend(); plt.gca().set_aspect("equal")

## Try it yourself
1. Check numerically that $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ at a few values of $z$.
2. Run `descend` with `eta=0.05` and `eta=0.07`. Which converge?
3. Change the step size `0.1` in section 6 to `0.3`. How many steps now? Try `0.35`.